# M12.0 — Frozen M11 baseline and acceptance contract

## A. Purpose and scientific freeze
Analyze saved **final M11 G1** predictions for `chirp_mass`, `total_mass`, `chi_eff`. This notebook defines an auditable baseline, not evidence of M12 improvement. The saved execution outputs contain the baseline metrics and completed artifact validations. This final correction preserves those computations and outputs; the appended verification and export cells have not yet been executed.

Freeze `M11_final_G1_500k`, train/val/cal/test = 400000/40000/30000/30000, seed 123, original row/source pairing, preprocessing, input normalization, train-only label scalers, architecture, and real-event protocol. H1/L1/V1, 4096 Hz, 4 s, SEOBNRv4_opt, full projection, G1 empirical PSD; inputs were z-scored before inference, per sample/detector, epsilon 1e-6. Do not normalize them again. Calibration and Mondrian remain untouched. No signals, embeddings, checkpoints, or models are loaded for inference.

M12.1 changes only the loss weighting using train-only label-space peripherality. M12.2 tests balanced sampling independently. Neither is implemented here. Test is a frozen final audit, not a tuning set; future selection uses validation under a preregistered hypothesis. The nine real events must never determine weights, bins, hyperparameters, stopping, or tolerances.

Sources inspected: [M11.10](../m11/M11_10_predictor_bias_diagnostics.ipynb), [M11.11](../m11/M11_11_final_cnn_evaluation.ipynb), [M11.13](../m11/M11_13_real_event_domain_transfer.ipynb), [M11.14](../m11/M11_14_real_events_lvk_comparison.ipynb), [M11.15](../m11/M11_15_result_interpretation.ipynb), [M11.15B](../m11/M11_15B_latent_domain_geometry.ipynb).
M11.10 uses pilot 20k G1 runs, **not this final baseline**. M11.15B latent geometry and its transductive synthetic scaling are not label-space Mahalanobis and must not define M12 weights.

The analysis reads saved artifacts and builds tables/JSON in memory. Only the final, manually invoked export cell writes the frozen baseline contract; it never overwrites an existing file.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import h5py
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open from the cbc_pe repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root, resolve_processed_artifact
from src.models.evaluate import regression_metrics, inverse_standardize

CONFIG_PATH = ROOT / "configs/experiments/train_m11_final_G1_500k.json"
config = json.loads(CONFIG_PATH.read_text())
# CBC_PE_DATA_ROOT overrides the historical config location; no hardcoded machine path.
DATA_ROOT = resolve_data_root(config_data_root=config["data_root"])
DATASET_ID = "M11_final_G1_500k"
LABELS = ["chirp_mass", "total_mass", "chi_eff"]
UNITS = ["Msun", "Msun", "dimensionless"]
EXPECTED_COUNTS = dict(train=400000, val=40000, cal=30000, test=30000)
RESULTS = DATA_ROOT / "results" / DATASET_ID
PRED_PATH = RESULTS / "m11_final_G1_500k_cal_test_predictions_embeddings.npz"
PATHS = {k: resolve_processed_artifact(data_root=DATA_ROOT, dataset_id=DATASET_ID,
            file_name=config["dataset"][k], allow_legacy_flat=False)
         for k in ("dataset_file", "split_file", "label_stats_file")}
REAL_DIR = RESULTS / "real_events/multi_event"

def require(ok, message):
    if not ok:
        raise ValueError(message)

def fingerprint(path, content=False):
    path = Path(path)
    if not path.is_file():
        return dict(path=str(path), status="missing")
    stat = path.stat()
    result = dict(path=str(path), status="present", size=stat.st_size, mtime_ns=stat.st_mtime_ns)
    if content:
        result["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
    return result

def array_hash(value):
    a = np.ascontiguousarray(value)
    return hashlib.sha256(str((a.dtype.str, a.shape)).encode() + a.tobytes()).hexdigest()

require(config["dataset"]["dataset_id"] == DATASET_ID, "Wrong final dataset config")
require(config["training"]["seed"] == 123, "Changed seed")
display(pd.DataFrame([dict(role=k, **fingerprint(p)) for k,p in
                     {"predictions": PRED_PATH, **PATHS}.items()]))

,role,path,status,size,mtime_ns
0,predictions,/data/vserrano/cbc_pe_data/results/M11_final_G...,present,15729924,1791196478720798398
1,dataset_file,/data/vserrano/cbc_pe_data/processed/M11_final...,present,98687856184,1790873275639179929
2,split_file,/data/vserrano/cbc_pe_data/processed/M11_final...,present,4003307,1790860193188182773
3,label_stats_file,/data/vserrano/cbc_pe_data/processed/M11_final...,present,3203077,1790860193198182373


## B. Saved predictions, targets, and provenance
Load test arrays only; calibration labels and embeddings remain unread. Reuse `src.models.evaluate` physical-unit metrics and inverse transform. Retain the M11.11 inverse-transform precision before promoting to float64 for reductions. Object-valued provenance follows M11.11's trusted local NPZ convention (`allow_pickle=True`); use only the repository's trusted saved artifact.

External paths embedded in the NPZ are historical provenance, not instructions to load arbitrary files. Resolve local artifacts through `src.paths`, check filenames and contracts, and preserve the original provenance. Checkpoint identity is recorded without deserializing it. File size/mtime is not a content identity proof; the output also hashes the actual ordered numeric prediction/target/index/scaler arrays and small contract files. Independently verifying the historical checkpoint bytes remains a provenance limitation.

In [2]:
require(PRED_PATH.is_file(), f"Unresolved saved predictions: {PRED_PATH}; do not regenerate")
with np.load(PRED_PATH, allow_pickle=True) as z:
    require(z["label_names"].tolist() == LABELS, "Prediction label order")
    require(set(z["available_splits"].tolist()) == {"cal", "test"}, "Unexpected saved splits")
    mean, std = (z[k].copy() for k in ("y_mean", "y_std"))
    truth_std, pred_std = (z[k].copy() for k in ("y_test", "pred_test"))
    idx = z["idx_test"].copy()
    saved_provenance = {k:z[k].item() for k in
        ("checkpoint_file", "dataset_path", "split_path", "label_stats_path",
         "model_config", "input_normalization")}
require(mean.shape == std.shape == (3,) and np.isfinite(mean).all()
        and np.isfinite(std).all() and (std > 0).all(), "Invalid saved train scaler")
require(truth_std.shape == pred_std.shape == (30000, 3), "Final test shape")
require(np.isfinite(truth_std).all() and np.isfinite(pred_std).all(), "Nonfinite predictions/targets")
require(idx.shape == (30000,) and np.issubdtype(idx.dtype, np.integer)
        and len(np.unique(idx)) == len(idx) and ((idx >= 0) & (idx < 500000)).all(), "Invalid test IDs")
require(saved_provenance["model_config"]["dataset_id"] == DATASET_ID
        and saved_provenance["model_config"]["label_names"] == LABELS, "Model provenance mismatch")
require(saved_provenance["input_normalization"] == config["input_normalization"] ==
        dict(enabled=False, already_normalized=True, mode="per_sample_per_detector_zscore", eps=1e-6),
        "Normalization contract mismatch")
for saved_key, local_key in (("dataset_path", "dataset_file"), ("split_path", "split_file"),
                             ("label_stats_path", "label_stats_file")):
    require(Path(saved_provenance[saved_key]).name == PATHS[local_key].name, "Artifact filename mismatch")
checkpoint_name = (f"{DATASET_ID}_SimpleCNN_ResidualDilated_"
                   f"{config['outputs']['checkpoint_tag']}_MSELoss_seed123_checkpoint.pt")
require(Path(saved_provenance["checkpoint_file"]).name == checkpoint_name, "Wrong checkpoint identity")
y_true = inverse_standardize(truth_std, mean, std).astype(np.float64)
y_pred = inverse_standardize(pred_std, mean, std).astype(np.float64)
require(np.isfinite(y_true).all() and np.isfinite(y_pred).all(), "Invalid physical values")
residual = y_pred - y_true
provenance = dict(saved=saved_provenance, config=fingerprint(CONFIG_PATH, True),
    prediction_file=fingerprint(PRED_PATH),
    numeric_sha256={k:array_hash(v) for k,v in dict(idx_test=idx, y_test=truth_std,
        pred_test=pred_std, y_mean=mean, y_std=std).items()},
    local_artifacts={k:fingerprint(p, k != "dataset_file") for k,p in PATHS.items()})
display(pd.DataFrame([dict(label=l, train_mean=mean[j], train_std=std[j]) for j,l in enumerate(LABELS)]))
print(json.dumps(saved_provenance, indent=2))

,label,train_mean,train_std
0,chirp_mass,37.463547,16.508060
1,total_mass,94.985977,34.703293
2,chi_eff,-0.000219,0.440338


{
  "checkpoint_file": "/data/vserrano/cbc_pe_data/models/checkpoints/M11_final_G1_500k/M11_final_G1_500k_SimpleCNN_ResidualDilated_M11_final_G1_inputzscore_resdilated_emb64_d124_train400k_MSELoss_seed123_checkpoint.pt",
  "dataset_path": "/data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/m11_final_G1_500k_trainio.h5",
  "split_path": "/data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/M11_final_G1_500k_splits_train400000_val40000_cal30000_test30000_seed123.npz",
  "label_stats_path": "/data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/M11_final_G1_500k_label_stats_train_only_train400000_val40000_cal30000_test30000_seed123.npz",
  "model_config": {
    "architecture": "SimpleCNN_ResidualDilated",
    "class_name": "SimpleCNN_ResidualDilated",
    "dataset_id": "M11_final_G1_500k",
    "dataset_format": "hdf5",
    "dataset_path": "/data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/m11_final_G1_500k_trainio.h5",
    "split_path": "/data/vserrano/cbc_pe_data/processed/M1

### Fixed split and metadata alignment
Read only HDF5 labels, split membership, and the named scalar SNR field. No strain arrays are accessed. Compare all four split memberships with the frozen split archive, and compare scaler values/train indices with the saved train-only scaler archive. Preserve the prediction archive's test row order. These dataset row IDs are not claimed to be astrophysical source IDs: a separate source-ID mapping was not established by the inspected analyses. Any future cross-dataset pairing requires that mapping; do not synthesize IDs.

Missing or inconsistent core artifacts stop the dependent analysis. No substitute dataset, split, scaler, or train reference is allowed. SNR alone may be marked unresolved while label-space metrics remain available.

In [3]:
for key, path in PATHS.items():
    require(path.is_file(), f"Unresolved {key}: {path}; restore frozen artifact, do not regenerate")
with np.load(PATHS["split_file"], allow_pickle=False) as z:
    split_ids = {s:z[f"{s}_idx"].copy() for s in EXPECTED_COUNTS}
    require(int(z["seed"].item()) == 123, "Split seed")
for s, ids in split_ids.items():
    require(ids.shape == (EXPECTED_COUNTS[s],) and np.issubdtype(ids.dtype, np.integer), f"Split {s}")
require(np.array_equal(np.sort(np.concatenate(list(split_ids.values()))), np.arange(500000)),
        "Splits do not partition G1 exactly")
require(np.array_equal(np.sort(idx), np.sort(split_ids["test"])), "Test membership changed")
with np.load(PATHS["label_stats_file"], allow_pickle=False) as z:
    require(z["label_names"].tolist() == LABELS, "Scaler labels")
    require(np.array_equal(z["y_mean"], mean) and np.array_equal(z["y_std"], std), "Scaler mismatch")
    require(np.array_equal(np.sort(z["train_idx"]), np.sort(split_ids["train"])), "Scaler train membership")
    require(int(z["seed"].item()) == 123, "Scaler seed")

snr, snr_issue = None, None
with h5py.File(PATHS["dataset_file"], "r") as h:
    require(h.attrs.get("experiment_id") == DATASET_ID, "Wrong HDF5 experiment")
    require(h.attrs.get("labels") == ",".join(LABELS), "HDF5 label order")
    splits = h["split"].asstr()[:]
    require(splits.shape == (500000,), "HDF5 split shape")
    for s, ids in split_ids.items():
        require(np.array_equal(np.flatnonzero(splits == s), np.sort(ids)), f"HDF5 split mismatch: {s}")
    require(h["y"].shape == (500000, 3), "Physical label shape")
    # Labels only, never X. In-memory indexing retains the original prediction order.
    labels_physical = np.asarray(h["y"][:], dtype=np.float64)
    require(np.isfinite(labels_physical).all(), "Nonfinite physical labels")
    require(np.allclose(labels_physical[idx], y_true, rtol=2e-6, atol=2e-5), "Target/row pairing mismatch")
    train_y = labels_physical[np.sort(split_ids["train"])].copy()
    del labels_physical, splits
    if ("final_network_snr" not in h or h.attrs.get("snr_low_frequency_cutoff") != 30.0
            or h.attrs.get("snr_high_frequency_cutoff") != 512.0):
        snr_issue = "Missing final_network_snr or unverified 30–512 Hz convention; no substitution"
    else:
        require(h["final_network_snr"].shape == (500000,), "SNR metadata shape")
        snr = np.asarray(h["final_network_snr"][:], dtype=float)[idx]
        require(np.isfinite(snr).all() and (snr >= 0).all(), "Invalid saved SNR")
print("Fixed membership and target alignment verified; SNR:", snr_issue or "verified")

Fixed membership and target alignment verified; SNR: verified


## C–D. Global metrics and Q10/Q90 tails
Physical-unit RMSE, MAE, R² and mean signed bias reuse M11.11 / `regression_metrics`: residual = prediction − truth; R² = 1 − SSE/SST. OLS fits **prediction against truth**, with an intercept, as in M11.10. Slope distance `abs(slope − 1)` is a newly named acceptance diagnostic. Constant truth makes OLS/R² undefined and stops this baseline.

For each label, Q10/Q90 are the evaluation truth's linearly interpolated 10th/90th percentiles. The lower subset is **truth ≤ Q10**, the upper **truth ≥ Q90**, including ties. Report each threshold, count, mean signed residual, and MAE. Positive lower-tail bias and negative upper-tail bias indicate inward compression; magnitude is assessed separately for both tails. These evaluation quantiles are **diagnostic subset definitions only**, never training weights, scaler fits, or Mahalanobis percentile fits. Freeze the thresholds and row pairing for later comparisons on this same test.

These inclusive tails are new M12 diagnostics, not a relabeling of M11.10 pilot deciles or M11.15 median-residual curves.

In [4]:
require((np.var(y_true, axis=0) > 0).all(), "Constant target: OLS/R2 undefined")
baseline = regression_metrics(y_true, y_pred, LABELS).set_index("label")
baseline = baseline[["RMSE", "MAE", "R2", "bias"]].copy()
baseline["n"] = len(idx)
baseline["unit"] = UNITS
for j, label in enumerate(LABELS):
    slope, intercept = np.polyfit(y_true[:, j], y_pred[:, j], 1)
    baseline.loc[label, ["slope", "intercept", "slope_distance_from_1"]] = [slope, intercept, abs(slope-1)]
    q10, q90 = np.quantile(y_true[:, j], [.1, .9])  # NumPy default linear interpolation
    for name, threshold, mask in (("lower", q10, y_true[:, j] <= q10),
                                  ("upper", q90, y_true[:, j] >= q90)):
        r = residual[mask, j]
        baseline.loc[label, [f"{name}_threshold", f"{name}_n", f"{name}_bias", f"{name}_MAE"]] = [
            threshold, len(r), r.mean(), np.abs(r).mean()]
for name in ("lower_n", "upper_n"):
    baseline[name] = baseline[name].astype(int)

## E. Train-only label-space Mahalanobis and fixed percentile bins
Reuse M11.15's exact formulation: physical train mean μ, population standard deviation σ (`ddof=0`), sample covariance Σ (`ddof=1`), C = Σ / outer(σ,σ). For z = (y−μ)/σ, d = sqrt(zᵀ C⁻¹ z). Use the inverse at full rank, otherwise the standardized-coordinate pseudoinverse with `rcond=1e-12`; record rank and condition number. This diagnostic standardization is distinct from, and does not replace, the saved label scaler.

**Explicit new M12 mapping:** evaluate d on all 400000 train labels and sort those distances. P(d) = 100 × count(train distances ≤ d) / 400000 (`searchsorted`, right side). Both metric statistics and empirical CDF are train-only. No evaluation labels enter either fit. Fixed bins are [0,20), [20,40), [40,60), [60,80), [80,100], with all ties assigned together; test counts need not be equal. Peripheral means P ≥ 80. Preserve empty bins; for n < 30, retain counts and return undefined conditional errors, following M11.15's support rule.

Historical M11.15 instead bins each test by its own distance quintiles and maps real percentiles against test distances. Its reported binned numbers are not directly comparable with these new train-CDF bins. We do not reproduce that mapping for M12 training or acceptance. No latent embeddings, local kNN density, calibration labels, or real-event coordinates enter this reference.

In [5]:
train_mu = train_y.mean(axis=0)
train_sigma = train_y.std(axis=0)
require((train_sigma > 0).all(), "Degenerate train scale")
train_cov = np.cov(train_y, rowvar=False, ddof=1)
scaled_cov = train_cov / np.outer(train_sigma, train_sigma)
rank = int(np.linalg.matrix_rank(scaled_cov))
precision = np.linalg.inv(scaled_cov) if rank == 3 else np.linalg.pinv(scaled_cov, rcond=1e-12)
def mahalanobis(values):
    z = (values - train_mu) / train_sigma
    squared = np.einsum("ni,ij,nj->n", z, precision, z)
    require(np.isfinite(squared).all() and (squared >= -1e-10).all(), "Invalid distance")
    return np.sqrt(np.maximum(squared, 0))
train_distance_sorted = np.sort(mahalanobis(train_y))
test_distance = mahalanobis(y_true)
test_percentile = 100 * np.searchsorted(train_distance_sorted, test_distance, side="right") / len(train_y)
PERCENTILE_EDGES = np.array([0., 20., 40., 60., 80., 100.])
MIN_COUNT = 30

def error_summary(r):
    n = len(r)
    return dict(n=n, sufficient_count=n >= MIN_COUNT,
        RMSE=float(np.sqrt(np.mean(r*r))) if n >= MIN_COUNT else None,
        MAE=float(np.mean(np.abs(r))) if n >= MIN_COUNT else None,
        bias=float(np.mean(r)) if n >= MIN_COUNT else None)

def conditional_table(values, edges, dimension):
    rows = []
    for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (values >= lo) & ((values <= hi) if b == len(edges)-2 else (values < hi))
        for j, label in enumerate(LABELS):
            rows.append(dict(label=label, dimension=dimension, bin=b, lower=float(lo), upper=float(hi),
                upper_inclusive=b == len(edges)-2, **error_summary(residual[mask, j])))
    return pd.DataFrame(rows)
mahal_table = conditional_table(test_percentile, PERCENTILE_EDGES, "train_mahalanobis_percentile")
for j, label in enumerate(LABELS):
    peripheral = error_summary(residual[test_percentile >= 80, j])
    for key in ("n", "RMSE", "MAE"):
        baseline.loc[label, f"peripheral_{key}"] = peripheral[key]
baseline["peripheral_n"] = baseline["peripheral_n"].astype(int)
geometry = dict(reference="M11 physical train labels only", n_train=len(train_y),
    mean=train_mu.tolist(), std_ddof0=train_sigma.tolist(), covariance_ddof1=train_cov.tolist(),
    scaled_covariance=scaled_cov.tolist(), precision=precision.tolist(), rank=rank,
    condition_number=float(np.linalg.cond(scaled_cov)), pinv_rcond=1e-12 if rank < 3 else None,
    percentile_mapping="100 * searchsorted(sorted train distances, d, side=right) / n_train",
    reference_distance_sha256=array_hash(train_distance_sorted),
    train_indices_sha256=array_hash(np.sort(split_ids["train"])), edges=PERCENTILE_EDGES.tolist())
display(mahal_table)

,label,dimension,bin,lower,upper,upper_inclusive,n,sufficient_count,RMSE,MAE,bias
0,chirp_mass,train_mahalanobis_percentile,0,0.0,20.0,False,5947,True,4.818542,3.544325,0.142543
1,total_mass,train_mahalanobis_percentile,0,0.0,20.0,False,5947,True,8.448100,6.161173,0.443228
2,chi_eff,train_mahalanobis_percentile,0,0.0,20.0,False,5947,True,0.152217,0.117195,-0.018903
3,chirp_mass,train_mahalanobis_percentile,1,20.0,40.0,False,6015,True,5.445517,3.841220,-0.197328
4,total_mass,train_mahalanobis_percentile,1,20.0,40.0,False,6015,True,9.539108,6.750256,0.214898
5,chi_eff,train_mahalanobis_percentile,1,20.0,40.0,False,6015,True,0.162147,0.124716,-0.003630
6,chirp_mass,train_mahalanobis_percentile,2,40.0,60.0,False,6034,True,5.503491,3.784653,-0.353895
7,total_mass,train_mahalanobis_percentile,2,40.0,60.0,False,6034,True,9.992115,7.062958,0.036872
8,chi_eff,train_mahalanobis_percentile,2,40.0,60.0,False,6034,True,0.180577,0.136525,-0.003656
9,chirp_mass,train_mahalanobis_percentile,3,60.0,80.0,False,5983,True,5.910271,4.015565,-0.220288


## F. Conditional errors by network SNR
Use exactly M11.15's HDF5 `final_network_snr`: final synthetic optimal network SNR in 30–512 Hz. Nominal bins [10,12.5), [12.5,15), [15,17.5), [17.5,20), [20,22.5), [22.5,25], with explicit under/overflow counts and errors. All rows remain in global metrics. The same n ≥ 30 rule applies.

`network_snr_lvk` is catalog SNR and is **not equivalent**. Target/requested SNR, a recomputed matched-filter SNR, or catalog SNR must never silently replace `final_network_snr`. Missing metadata or unverified band attributes yields an unresolved SNR section, not an alternate definition. No SNR-dependent training weights are fitted here.

In [6]:
SNR_EDGES = np.array([10., 12.5, 15., 17.5, 20., 22.5, 25.])
snr_table = pd.DataFrame()
snr_outside = pd.DataFrame()
if snr is not None:
    snr_table = conditional_table(snr, SNR_EDGES, "final_network_snr")
    snr_outside = pd.DataFrame([dict(label=label, region=region, **error_summary(residual[mask, j]))
        for region, mask in (("below_10", snr < 10), ("above_25", snr > 25))
        for j, label in enumerate(LABELS)])
    display(snr_table)
    display(snr_outside)
else:
    print("UNRESOLVED SNR:", snr_issue)

,label,dimension,bin,lower,upper,upper_inclusive,n,sufficient_count,RMSE,MAE,bias
0,chirp_mass,final_network_snr,0,10.0,12.5,False,5034,True,7.920882,5.534133,-0.007537
1,total_mass,final_network_snr,0,10.0,12.5,False,5034,True,14.658759,10.682009,-0.234782
2,chi_eff,final_network_snr,0,10.0,12.5,False,5034,True,0.253620,0.197698,-0.013934
3,chirp_mass,final_network_snr,1,12.5,15.0,False,5034,True,6.609982,4.610422,0.197617
4,total_mass,final_network_snr,1,12.5,15.0,False,5034,True,11.941546,8.775706,0.318537
5,chi_eff,final_network_snr,1,12.5,15.0,False,5034,True,0.218976,0.167396,-0.009307
6,chirp_mass,final_network_snr,2,15.0,17.5,False,4978,True,5.713511,3.988509,0.247114
7,total_mass,final_network_snr,2,15.0,17.5,False,4978,True,9.961887,7.370762,0.444198
8,chi_eff,final_network_snr,2,15.0,17.5,False,4978,True,0.188105,0.144612,-0.005575
9,chirp_mass,final_network_snr,3,17.5,20.0,False,4939,True,5.120122,3.538002,0.147327


,label,region,n,sufficient_count,RMSE,MAE,bias
0,chirp_mass,below_10,0,False,None,None,None
1,total_mass,below_10,0,False,None,None,None
2,chi_eff,below_10,0,False,None,None,None
3,chirp_mass,above_25,0,False,None,None,None
4,total_mass,above_25,0,False,None,None,None
5,chi_eff,above_25,0,False,None,None,None


## G. Concise baseline table
One row per scientific label, in physical units. RMSE/MAE/bias/intercept/tail errors carry the label unit; R² and slope are dimensionless. Conditional tables above retain all bin counts; no cross-label average combines unlike units. The saved outputs retain the values from the completed analysis run.

In [7]:
baseline_table = baseline.reset_index()
with pd.option_context("display.max_columns", None, "display.width", 220):
    display(baseline_table.round(6))

,label,RMSE,MAE,R2,bias,n,unit,slope,intercept,slope_distance_from_1,lower_threshold,lower_n,lower_bias,lower_MAE,upper_threshold,upper_n,upper_bias,upper_MAE,peripheral_n,peripheral_RMSE,peripheral_MAE
0,chirp_mass,5.768810,3.924969,0.877664,0.154418,30000,Msun,0.836533,6.278150,0.163467,16.425760,3000,2.780938,2.819798,60.922143,3000,-5.106718,5.636004,6021,6.942170,4.435196
1,total_mass,10.408432,7.403433,0.909414,0.333935,30000,Msun,0.863671,13.293932,0.136329,48.027045,3000,7.369195,7.852886,141.834158,3000,-9.533415,10.803681,6021,12.658432,9.130486
2,chi_eff,0.190098,0.142680,0.813688,-0.004617,30000,dimensionless,0.782103,-0.005465,0.217897,-0.600999,3000,0.192597,0.202131,0.597352,3000,-0.166768,0.172834,6021,0.241423,0.181438


## H. Acceptance contract — provisional until uncertainty is assessed
A later candidate must pass the artifact/split/pairing/normalization contract before any scientific comparison. Evaluate each label independently with identical test rows, targets, tail thresholds, train reference, percentile mapping and SNR bins. Compare candidate minus M11; negative deltas are favorable for errors and distances.

**Candidate selection must use validation**, under a preregistered hypothesis. Do not repeatedly select candidates, tune hyperparameters, or revise acceptance thresholds against the frozen test. Reserve test for the final audit after validation-based selection. This notebook does not generate validation predictions.

| Criterion | Comparison | Contract status |
|---|---|---|
| Regression compression | Δ abs(slope − 1) | Must improve for the preregistered target labels; report all three |
| Both tails | Δ abs(lower bias), Δ abs(upper bias), plus tail MAE | Assess both separately; do not cancel opposite signs |
| Periphery | Δ RMSE and Δ MAE for train percentile ≥ 80 | Require supported improvement; inspect all fixed bins |
| Global guardrails | Δ RMSE and Δ MAE; report Δ R² and signed bias | Maximum allowed degradation **TBD**, per label and metric |
| SNR robustness | Conditional error deltas, counts, under/overflow | No hidden low-SNR or sparse-bin failure; tolerances **TBD** |
| Final real-event audit | Frozen paired M11/M12 discrepancies to the same LVK references | Separate descriptive audit after synthetic selection; never optimize on nine events |

All numerical degradation tolerances and minimum meaningful improvements remain **provisional (`null`)**. There is no automatic pass/fail while they are unset. Before running expensive experiments, preregister the hypothesis, target labels, tradeoff rule, and uncertainty procedure. Assess paired uncertainty using identical source/sample IDs (group by source if repeated sources exist), and training-seed variation where authorized. Do not run resampling or additional seeds here. Set tolerances before inspecting candidate test/real outcomes; report uncertainty rather than treating point differences as established improvement. The unresolved source-ID mapping limits independence assumptions. Test-based baseline tail diagnostics cannot become training weights.

Changing code alone does not establish improvement. Keep Mondrian calibration and real-event preprocessing fixed; any missing frozen state stays missing.

In [8]:
acceptance_contract = dict(status="provisional_not_evaluable", baseline="M11_final_G1_500k",
    candidate_minus_baseline=True, labels=LABELS,
    target_labels=None, minimum_improvements=None,
    degradation_tolerances={label:dict(global_RMSE=None, global_MAE=None,
        lower_tail_MAE=None, upper_tail_MAE=None, peripheral_RMSE=None,
        peripheral_MAE=None, snr_conditional_error=None) for label in LABELS},
    primary_diagnostics=["slope_distance_from_1", "abs(lower_bias)", "abs(upper_bias)",
                         "peripheral_RMSE", "peripheral_MAE"],
    peripheral_definition="train_mahalanobis_percentile >= 80",
    uncertainty_status="not assessed", automatic_decision=None,
    real_events_role="separate final descriptive audit; prohibited for optimization")

## I. Historical real-event comparisons — saved artifacts only
M11.14 preserves 14 candidates, nine successes and five skips. Read `event_status.csv`, `event_predictions.csv`, `per_event_comparison.csv`, `aggregate_summary.csv`, and `provenance.json` under `results/M11_final_G1_500k/real_events/multi_event/`. Preserve detector-frame masses and dimensionless spin; LVK central estimates are uncertain posterior references, not ground truth. Conservative intervals are the historical primary policy; efficient is secondary. Do not recalibrate or reinterpret discrepancy as synthetic RMSE.

The saved nominal manifest specifies PSD [−1024,−640] s, 8 s segments, center offset 0, no fallback windows, and z-score epsilon 1e-6. M11.13's GW170814 uses a separate GWTC-1 posterior anchor and is excluded from the nine-event aggregates. Its saved `real_events/GW170814/m10_vs_m11_summary.csv`, `conformal_status.csv`, and `provenance.json` may be inspected separately; missing frozen interval states must not be reconstructed. No historical sensitivity sweep or inference is rerun.

M11.15 reports regression compression and peripheral errors descriptively; M11.15B explores latent-domain compatibility separately. Neither supplies acceptance thresholds for M12. This section displays saved comparisons without recomputing historical numbers.

In [9]:
real_files = {name:REAL_DIR / f"{name}.csv" for name in
              ("event_status", "event_predictions", "per_event_comparison", "aggregate_summary")}
real_manifest_path = REAL_DIR / "provenance.json"
real_history = dict(status="unresolved", files={k:fingerprint(p, True) for k,p in real_files.items()},
                    manifest=fingerprint(real_manifest_path, True))
missing_real = [str(p) for p in [*real_files.values(), real_manifest_path] if not p.is_file()]
if missing_real:
    real_history["missing"] = missing_real
    print("UNRESOLVED historical comparisons:", missing_real)
else:
    saved_tables = {k:pd.read_csv(p) for k,p in real_files.items()}
    manifest = json.loads(real_manifest_path.read_text())
    status = saved_tables["event_status"]
    paired = saved_tables["per_event_comparison"]
    points = saved_tables["event_predictions"]
    require(not status.event.duplicated().any(), "Duplicate event status")
    cohort = status.loc[status.processing_status.eq("success"), "event"].tolist()
    require(len(status) == 14 and len(cohort) == 9
            and status.processing_status.eq("skipped").sum() == 5, "Historical cohort changed")
    require("GW170814" not in cohort, "Anchor mixed into cohort")
    require(not paired.duplicated(["event", "label"]).any()
            and len(paired) == 27 and set(paired.event) == set(cohort), "Incomplete comparison pairing")
    require(not points.duplicated(["event", "model", "label"]).any(), "Duplicate predictions")
    for label in LABELS:
        require(set(paired.loc[paired.label.eq(label), "event"]) == set(cohort), "Missing label pairs")
        frame = "dimensionless" if label == "chi_eff" else "detector"
        require(points.loc[points.label.eq(label), "frame"].eq(frame).all(), "Historical frame changed")
    for model in ("M10", "M11"):
        check = paired.merge(points.loc[points.model.eq(model), ["event", "label", "prediction"]],
                             on=["event", "label"], validate="one_to_one")
        require(len(check) == 27 and np.isfinite(check[["prediction", "lvk_central"]]).all().all(),
                "Incomplete historical point predictions")
        require(np.allclose(check.prediction, check[f"prediction_{model}"])
                and np.allclose(abs(check.prediction-check.lvk_central), check[f"absolute_discrepancy_{model}"]),
                "Historical discrepancy mismatch")
    require(manifest["nominal_policy"] == dict(psd_window=[-1024., -640.], center_offset=0.,
        psd_segment_duration=8., fallback_windows=[], zscore_eps=1e-6), "Real nominal policy changed")
    require(manifest["primary_policy"] == "conservative", "Historical interval policy changed")
    require(Path(manifest["checkpoints"]["M11"]).name == checkpoint_name, "Real checkpoint mismatch")
    real_history.update(status="loaded_saved_only", cohort=cohort, manifest_content=manifest,
                        tables={k:v.to_dict("records") for k,v in saved_tables.items()})
    display(saved_tables["aggregate_summary"])
    display(paired[["event", "label", "lvk_central", "prediction_M10", "prediction_M11",
                    "absolute_discrepancy_M10", "absolute_discrepancy_M11"]])
anchor_dir = RESULTS / "real_events/GW170814"
real_history["separate_anchor_files"] = {name:fingerprint(anchor_dir / name, True) for name in
    ("m10_vs_m11_summary.csv", "conformal_status.csv", "provenance.json")}

,label,n_valid_events,fraction_M11_closer,fraction_equal,fraction_M11_further,median_absolute_discrepancy_M10,median_mondrian_width_M10,fraction_intervals_overlapped_M10,median_overlap_fraction_lvk_M10,median_absolute_discrepancy_M11,median_mondrian_width_M11,fraction_intervals_overlapped_M11,median_overlap_fraction_lvk_M11
0,chirp_mass,9,0.666667,0.0,0.333333,2.453911,16.720632,0.888889,0.678606,1.987950,17.083339,1.000000,1.000000
1,total_mass,9,0.777778,0.0,0.222222,6.351077,25.056439,0.888889,0.920607,3.873451,27.696938,0.888889,0.849341
2,chi_eff,9,0.888889,0.0,0.111111,0.307677,0.594867,1.000000,0.637699,0.087100,0.608739,1.000000,0.990113


,event,label,lvk_central,prediction_M10,prediction_M11,absolute_discrepancy_M10,absolute_discrepancy_M11
0,GW200202_154313,chirp_mass,8.1641,10.618011,15.163832,2.453911,6.999732
1,GW200202_154313,total_mass,19.1622,27.853656,39.871470,8.691456,20.709270
2,GW200202_154313,chi_eff,0.0400,0.433547,0.166536,0.393547,0.126536
3,GW200316_215756,chirp_mass,10.6750,12.507020,13.909145,1.832020,3.234145
4,GW200316_215756,total_mass,25.8640,32.079116,35.095554,6.215116,9.231554
5,GW200316_215756,chi_eff,0.1300,0.071046,0.059093,0.058954,0.070907
6,GW191215_223052,chirp_mass,24.8400,16.529647,23.218558,8.310353,1.621442
7,GW191215_223052,total_mass,58.4550,40.197370,54.644449,18.257630,3.810551
8,GW191215_223052,chi_eff,-0.0400,-0.483656,-0.377585,0.443656,0.337585
9,GW200129_065458,chirp_mass,32.0960,30.129896,31.241331,1.966104,0.854669


### Reusable machine-readable schema (version 1)
`baseline_payload` / `baseline_json` is the in-memory output contract for M12 comparisons. Top-level fields identify experiment, labels/units, evaluation split, provenance, definitions, reference statistics, baseline rows, conditional rows, acceptance status, historical references, and unresolved dependencies. Each baseline row is keyed by label; conditional rows by label/dimension/bin, including numeric bounds and counts. Undefined values serialize as JSON `null`, never NaN/Infinity. This schema cell creates no output file; the final export cell persists this representation only when manually invoked.

Later experiments must reuse the ordered IDs/target hashes, scaler, diagnostic thresholds, and train-CDF reference identity. The full sorted train-distance array remains in memory as `train_distance_sorted`; it can be reconstructed only from the identical frozen train labels and recorded formula, not refitted on a candidate evaluation set. Its hash detects a changed reference. The schema embeds small statistics and identities rather than duplicating large datasets. Extend the schema version if definitions change; never overwrite this baseline with candidate values.

In [10]:
unresolved = [
    "Uncertainty and numerical acceptance tolerances are provisional; no pass/fail decision",
    "Source-ID mapping beyond frozen dataset row IDs not established; review before grouped uncertainty",
    "Historical checkpoint content identity is not independently verified by deserialization or content hash",
]
if snr_issue:
    unresolved.append(snr_issue)
if real_history["status"] != "loaded_saved_only":
    unresolved.append("Historical real-event comparison artifacts incomplete")

def json_safe(value):
    if isinstance(value, dict):
        return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

baseline_payload = json_safe(dict(schema_version=1, experiment="M12.0", model="frozen M11",
    dataset_id=DATASET_ID, split="test", labels=LABELS, units=dict(zip(LABELS, UNITS)),
    provenance=provenance, split_counts=EXPECTED_COUNTS,
    definitions=dict(residual="prediction - truth", slope="OLS prediction against truth with intercept",
        R2="1 - SSE/SST", tails="evaluation true-label linear Q10/Q90; <= Q10, >= Q90; diagnostic only",
        conditional_min_count=MIN_COUNT, percentile_bins="left inclusive; final upper inclusive",
        snr="HDF5 final_network_snr, optimal final network SNR 30–512 Hz",
        snr_edges=SNR_EDGES.tolist(), historical_mahalanobis_mapping="test-based; not reused"),
    geometry=geometry, baseline=baseline_table.to_dict("records"),
    conditional_mahalanobis=mahal_table.to_dict("records"),
    conditional_snr=snr_table.to_dict("records"), snr_outside=snr_outside.to_dict("records"),
    snr_status=snr_issue or "available", acceptance=acceptance_contract,
    historical_real=real_history, unresolved=unresolved,
    versions=dict(python=sys.version, numpy=np.__version__, pandas=pd.__version__, h5py=h5py.__version__)))
baseline_json = json.dumps(baseline_payload, indent=2, allow_nan=False)
print("Machine-readable baseline prepared in memory; no files written.")

Machine-readable baseline prepared in memory; no files written.


## Verified validations and remaining limitations
The saved run completed prediction/scaler checks, exact split membership and HDF5 target alignment, and the synthetic SNR field/band checks. Saved outputs contain global and tail metrics, train-only label-space Mahalanobis summaries, SNR summaries, and the historical nine-event comparisons. Historical cohort, frame, nominal-policy, and prediction/discrepancy consistency checks completed, and the JSON payload was prepared in memory. These are existing execution results, not newly recalculated validations.

Remaining limitations are independent checkpoint content verification, source-ID mapping beyond frozen dataset row IDs, and uncertainty assessment/numerical acceptance tolerances. File size/mtime and checkpoint filenames do not prove checkpoint content identity. The new bin assertions and export below remain unexecuted; existing outputs do not establish that these new cells passed. Missing artifacts must remain unresolved, never regenerated here.

Python 3.10 compatibility is checked statically using its syntax grammar; no code cells or dependency/runtime checks are executed during this correction. Existing computations and their outputs are unchanged.

With the completed analysis still available in the kernel, manually run only **Verify existing conditional bins**, then **Export frozen baseline contract**. The export rechecks the assertions before writing. Saved notebook outputs alone do not restore kernel variables: if the kernel was restarted, the preceding analysis state must first be restored by explicitly running its prerequisite sections in order in `cbc_torch`; do not use Run All. No training, inference, validation prediction generation, or historical notebook execution is needed.


### Verify existing conditional bins
Run this cell manually using the existing analysis state. It checks percentile bounds, one bin assignment per test row, and each label's complete conditional counts without recalculating any metric. SNR counts include the explicit underflow/overflow regions; an unresolved SNR section remains unresolved.

In [11]:
def verify_baseline_bins():
    require(test_percentile.shape == (len(idx),)
            and np.isfinite(test_percentile).all()
            and ((test_percentile >= 0) & (test_percentile <= 100)).all(),
            "Mahalanobis percentiles must be finite and in [0, 100]")
    for values, edges, table, outside in (
            (test_percentile, PERCENTILE_EDGES, mahal_table, None),
            (snr, SNR_EDGES, snr_table, snr_outside)):
        if values is None:
            continue
        assignments = np.zeros(len(idx), dtype=int)
        require(len(table) == len(LABELS) * (len(edges)-1)
                and set(table.label) == set(LABELS)
                and not table.duplicated(["label", "bin"]).any(),
                "Incomplete or duplicate conditional bins")
        for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
            mask = (values >= lo) & ((values <= hi) if b == len(edges)-2 else (values < hi))
            assignments += mask
            rows = table.loc[table.bin.eq(b)]
            require(len(rows) == len(LABELS) and (rows.n == int(mask.sum())).all(),
                    "Conditional bin counts disagree with sample assignments")
        if outside is not None:
            require(len(outside) == 2 * len(LABELS) and set(outside.label) == set(LABELS)
                    and not outside.duplicated(["label", "region"]).any(),
                    "Incomplete SNR outside-bin counts")
            for region, mask in (("below_10", values < edges[0]), ("above_25", values > edges[-1])):
                assignments += mask
                rows = outside.loc[outside.region.eq(region)]
                require(len(rows) == len(LABELS) and (rows.n == int(mask.sum())).all(),
                        "SNR outside-bin count mismatch")
        require((assignments == 1).all(), "Each test sample must have exactly one bin assignment")
        counts = table.groupby("label").n.sum().reindex(LABELS)
        if outside is not None:
            counts = counts + outside.groupby("label").n.sum().reindex(LABELS)
        require((counts == len(idx)).all(), "Conditional counts must cover the full test per label")

verify_baseline_bins()
print("Percentile bounds, unique assignments, and complete conditional counts verified.")

Percentile bounds, unique assignments, and complete conditional counts verified.


### Export frozen baseline contract — explicitly invoke this cell
Run the following cell manually only after reviewing the baseline and its remaining limitations. It exports the existing `baseline_payload` through `json_safe`, without adding prediction, target, or reference-distance arrays, to `notebooks/m12/outputs/m12_0_baseline_contract.json`.

This is a **frozen, versioned scientific baseline** (`schema_version=1`), including the explicitly provisional acceptance tolerances. Preserve it for subsequent M12 comparisons. Exclusive creation raises `FileExistsError` if the destination exists; do not delete or overwrite it to rerun this cell. Any approved revision requires a separately versioned artifact and documented provenance, not silent replacement. The parent directory is created only when this cell is invoked.

In [12]:
# Explicit manual export; no scientific metrics are recomputed.
verify_baseline_bins()
export_text = json.dumps(json_safe(baseline_payload), indent=2, allow_nan=False)
export_path = ROOT / "notebooks/m12/outputs/m12_0_baseline_contract.json"
export_path.parent.mkdir(parents=True, exist_ok=True)
with export_path.open("x", encoding="utf-8") as stream:
    stream.write(export_text + "\n")
print("Frozen baseline contract exported:", export_path)

Frozen baseline contract exported: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks/m12/outputs/m12_0_baseline_contract.json
